In [130]:
import re
import math
from collections import Counter

import pandas as pd
import numpy as np

In [131]:
df = pd.read_csv("data/Indonesian_Food_Recipes.csv")
df.head()

,Title,Ingredients,Steps,Loves,URL,Category,Title Cleaned,Total Ingredients,Ingredients Cleaned,Total Steps
0,Ayam Woku Manado,1 Ekor Ayam Kampung (potong 12)--2 Buah Jeruk ...,1) Cuci bersih ayam dan tiriskan. Lalu peras j...,1,https://cookpad.com/id/resep/4473027-ayam-woku...,ayam,ayam woku manado,14,"ayam kampung potong , jeruk nipis , garam , ku...",7
1,Ayam goreng tulang lunak,1 kg ayam (dipotong sesuai selera jangan kecil...,"1) Haluskan bumbu2nya (BaPut, ketumbar, kemiri...",1,https://cookpad.com/id/resep/4471956-ayam-gore...,ayam,ayam goreng tulang lunak,11,"ayam dipotong , serai , daun jeruk , bawang pu...",5
2,Ayam cabai kawin,1/4 kg ayam--3 buah cabai hijau besar--7 buah ...,1) Panaskan minyak di dalam wajan. Setelah min...,2,https://cookpad.com/id/resep/4473057-ayam-caba...,ayam,ayam cabai kawin,10,"ayam , cabai hijau , cabai merah rawit , bawan...",3
3,Ayam Geprek,250 gr daging ayam (saya pakai fillet)--Secuku...,1) Goreng ayam seperti ayam krispi\n2) Ulek se...,10,https://cookpad.com/id/resep/4473023-ayam-geprek,ayam,ayam geprek,7,"daging ayam fillet , gula garam , tepung ayam ...",3
4,Minyak Ayam,400 gr kulit ayam & lemaknya--8 siung bawang p...,1) Cuci bersih kulit ayam. Sisihkan\n2) Ambil ...,4,https://cookpad.com/id/resep/4427438-minyak-ayam,ayam,minyak ayam,5,"kulit ayam & lemaknya , bawang putih , cincang...",6


In [132]:
COLS = [
    "Title",
    "Title Cleaned",
    "Category",
    "Ingredients Cleaned",
    "Steps",
    "Loves",
    "URL",
    "Total Ingredients",
    "Total Steps",
]

recipes = df[COLS].copy()

In [133]:
recipes["ingredient_list"] = (
    recipes["Ingredients Cleaned"]
    .astype(str)
    .str.split(",")
)

## Stopwords & Noise Words

In [134]:
STOPWORDS_INGREDIENT = {
    # proses memasak
    "haluskan", "dihaluskan", "halus",
    "iris", "diiris",
    "potong", "dipotong",
    "cincang", "dicincang",
    "geprek", "digeprek",
    "rebus", "direbus", "merebus",
    "goreng", "digoreng",
    "kukus", "dikukus",
    "blender", "diblender",
    "tumis",
    "campur", "dicampur",
    "larutkan", "dilarutkan",
    "ungkep", "mengungkep",
    "panggang", "dipanggang",

    # bentuk / ukuran
    "tipis", "tebal", "kasar", "dadu", "kotak", "memanjang",

    # instruksi
    "ambil", "buang", "pisahkan", "bersihkan", "dibersihkan", "sisa",

    # kata umum
    "utk", "untuk", "tuk",
    "secukupnya",
    "sesuai", "selera",
    "kuah", "bumbu",
    "dirajang",
    "baluran", "olesan", "cocolan",
    "dg", "dgn", "nya", "ny", "me", "pake", "pke",
    "menumis", "untuk menumis", "untuk tumisan",
    "pelengkap", "hiasan", "taburan", "garnish",
    "sesendok", "sedikit", "agak", "kurang",
}

In [135]:
NOISE_WORDS = {
    "hehe", "hehehe", "wkwk", "wkwkwk",
    "suka", "ga", "gak", "ngga", "nggak",
    "kalo", "kalau",
    "aja", "aj", "banget", "loh", "nih",
    "bisa", "anak", "ikutan",
    "sesuka", "sesukanya",
}

## Ingredient Mapping & Normalization

In [136]:
ingredient_mapping = {
    # CABAI
    "cabe": "cabai",
    "cabe rawit": "cabai",
    "cabe rawit merah": "cabai",
    "cabai merah": "cabai",
    "cabai keriting": "cabai",
    "cabai rawit": "cabai",

    # AYAM
    "ayam kampung": "ayam",
    "ayam kampung potong": "ayam",
    "ayam broiler": "ayam",
    "ayam fillet": "ayam",
    "ayam dada fillet": "ayam",
    "dada ayam": "ayam",
    "paha ayam": "ayam",

    # BAWANG
    "bawang merah iris": "bawang merah",
    "bawang putih cincang": "bawang putih",

    # JERUK
    "air jeruk nipis": "jeruk nipis",

    # SANTAN
    "santan kelapa": "santan",
    "santan kara": "santan",

    # PENYEDAP
    "royco": "penyedap",
    "masako": "penyedap",
    "kaldu ayam": "penyedap",
}

In [137]:
def normalize_common_variants(ingredient):
    ingredient = ingredient.lower()
    ingredient = re.sub(r"\d+", "", ingredient)

    ingredient = ingredient.replace("cabe", "cabai")

    if "minyak wijen" in ingredient:
        return "minyak wijen"
    if "minyak zaitun" in ingredient:
        return "minyak zaitun"
    if ingredient.startswith("minyak"):
        return "minyak goreng"
    if ingredient.startswith("air "):
        return "air"
    if "garam" in ingredient:
        return "garam"
    if "gula pasir" in ingredient:
        return "gula"
    if any(k in ingredient for k in ["royco", "masako", "kaldu bubuk", "penyedap"]):
        return "penyedap"
    if "bawang putih" in ingredient:
        return "bawang putih"
    if "bawang merah" in ingredient:
        return "bawang merah"
    if "telur ayam" in ingredient:
        return "telur"
    if "telur" in ingredient:
        return "telur"
    if "ayam" in ingredient:
        return "ayam"

    return ingredient

In [138]:
def clean_ingredient(ingredient):
    ingredient = str(ingredient).lower().strip()
    ingredient = re.sub(r"\d+", "", ingredient)           # hapus angka
    ingredient = re.sub(r"\(.*?\)", "", ingredient)       # hapus kurung
    ingredient = re.sub(r"[^a-zA-Z\s]", " ", ingredient)  # hanya huruf
    ingredient = re.sub(r"\s+", " ", ingredient)          # spasi berlebih
    return ingredient.strip()

In [139]:
def remove_stopwords(ingredient):
    words = ingredient.split()
    words = [w for w in words if w not in STOPWORDS_INGREDIENT]
    return " ".join(words)

In [140]:
def normalize_ingredient(ingredient):
    ingredient = clean_ingredient(ingredient)
    ingredient = remove_stopwords(ingredient)
    ingredient = normalize_common_variants(ingredient)
    ingredient = ingredient.strip()

    if ingredient in ingredient_mapping:
        ingredient = ingredient_mapping[ingredient]

    words = ingredient.split()

    if len(words) > 5:
        return ""
    if any(w in NOISE_WORDS for w in words):
        return ""

    return ingredient

## Apply Normalization

In [141]:
recipes["ingredient_list"] = recipes["ingredient_list"].apply(
    lambda ingredients: [
        normalize_ingredient(i)
        for i in ingredients
        if normalize_ingredient(i) != ""
    ]
)

In [142]:
recipes["ingredient_list"] = recipes["ingredient_list"].apply(
    lambda ingredients: [i for i in ingredients if len(i) > 0]
)

## Standardize & Filter Rare Ingredients

In [143]:
all_ingredients = []
for ingredients in recipes["ingredient_list"]:
    all_ingredients.extend(ingredients)

unique_ingredients = set(all_ingredients)
print("Jumlah ingredient unik:", len(unique_ingredients))

Jumlah ingredient unik: 16789


In [144]:
def standardize_ingredients(ingredient_list):
    return [
        ingredient_mapping.get(ing.strip(), ing.strip())
        for ing in ingredient_list
    ]

In [145]:
recipes["ingredient_list"] = recipes["ingredient_list"].apply(standardize_ingredients)

In [146]:
ingredient_counter = Counter()
for ingredients in recipes["ingredient_list"]:
    ingredient_counter.update(ingredients)

MIN_FREQ = 3
valid_ingredients = {
    ing for ing, freq in ingredient_counter.items()
    if freq >= MIN_FREQ
}

In [147]:
def process_ingredients(ingredients):
    result = []

    for i in ingredients:
        ing = normalize_ingredient(i)

        if ing != "":
            result.append(ing)

    return result


recipes["ingredient_list"] = (
    recipes["ingredient_list"]
    .apply(process_ingredients)
)

## Hitung IDF

In [148]:
ingredient_df = Counter()
for ingredients in recipes["ingredient_list"]:
    for ingredient in set(ingredients):
        ingredient_df[ingredient] += 1

In [149]:
N = len(recipes)

idf_dict = {
    ingredient: math.log(N / df_count)
    for ingredient, df_count in ingredient_df.items()
}

In [150]:
idf_df = pd.DataFrame({
    "ingredient": list(idf_dict.keys()),
    "idf": list(idf_dict.values()),
})

idf_df.sort_values(by="idf", ascending=False).head(20)

,ingredient,idf
16788,udang kulkas ja,9.612132
16787,bsngmer,9.612132
16786,kecap pedis,9.612132
16770,biting tusukan sate,9.612132
16769,jeruk limau penghilang amis,9.612132
16768,margarin loyang,9.612132
16767,unt pencelup,9.612132
16766,udang sebentar berubah warna,9.612132
16765,santai kara,9.612132
16764,kubis cina,9.612132


## Statistik Akhir

In [151]:
ingredient_counter = Counter()
for ingredients in recipes["ingredient_list"]:
    ingredient_counter.update(ingredients)

print("Jumlah ingredient unik:", len(ingredient_counter))

Jumlah ingredient unik: 16789


In [152]:
rare = [ing for ing, count in ingredient_counter.items() if count == 1]
print("Jumlah ingredient muncul 1x:", len(rare))
print(rare[:50])

Jumlah ingredient muncul 1x: 12844
['lidi tusuk gigi mengapit daun', 'merica takar', 'hintalu jaruk', 'bawang lupa', 'sambel penyet', 'mangkok makaroni', 'timun biji korek api', 'maizena larutlan air', 'arak beras', 'saos hoisin', 'larutan tapioka', 'tepung pedas sajiku', 'cabai keritinh', 'margarin unsalted', 'baham sambal', 'perbandingan', 'saos lea perrins', 'api', 'ngo hiang', 'tetes suclarose stevia drop', 'tepung panir membalur', 'cabai lalap', 'keping asam kering', 'nasi hainan', 'rebusan chicken hainan', 'saus hainan', 'cabai hujau', 'ktumbar bbuk', 'laus', 'daun salm', 'bhn sambl ulek', 'terasi bkar', 'jeruk nipis jeruk limau', 'rosemary thyme kering', 'santan santan kemasan air', 'siun bawang meraih', 'tepung sagu aq sagu tani', 'cabai merah bole ditambah', 'bawamg putih', 'keju parut chedar', 'kembang tahu kering', 'jamur campion', 'saus kikoman', 'buncis lalapan', 'bawang putuh', 'bbrp nanas', 'daun jaruk', 'blm', 'tambahkan pedas', 's butter']


In [153]:
ingredient_counter.most_common(50)

[('bawang putih', 13437),
 ('garam', 13314),
 ('bawang merah', 9853),
 ('air', 6257),
 ('cabai', 6135),
 ('minyak goreng', 5278),
 ('gula', 5010),
 ('penyedap', 4952),
 ('telur', 3971),
 ('jahe', 3254),
 ('ayam', 3227),
 ('daun salam', 3079),
 ('kecap manis', 2923),
 ('tomat', 2884),
 ('kemiri', 2646),
 ('daun jeruk', 2586),
 ('kunyit', 2462),
 ('lada', 2397),
 ('bawang bombai', 2359),
 ('serai', 2308),
 ('lengkuas', 2187),
 ('merica', 1875),
 ('ketumbar', 1736),
 ('daun bawang', 1276),
 ('wortel', 1235),
 ('udang', 1202),
 ('merica bubuk', 1178),
 ('batang daun bawang', 1070),
 ('tepung terigu', 1060),
 ('papan tempe', 976),
 ('cabai rawit merah', 959),
 ('kecap', 926),
 ('saus tiram', 904),
 ('daging sapi', 898),
 ('daging kambing', 897),
 ('kentang', 817),
 ('gula merah', 815),
 ('cabai merah keriting', 772),
 ('jeruk nipis', 771),
 ('biji ketumbar', 728),
 ('tahu', 727),
 ('santan', 708),
 ('pala', 674),
 ('saos tiram', 663),
 ('bawang', 627),
 ('tempe', 620),
 ('tahu putih', 615),

In [154]:
recipes["ingredient_count"] = recipes["ingredient_list"].apply(len)
recipes["ingredient_count"].describe()

count    14945.000000
mean        12.421947
std          4.992762
min          1.000000
25%          9.000000
50%         12.000000
75%         15.000000
max         70.000000
Name: ingredient_count, dtype: float64

In [155]:
sum("" in ingredients for ingredients in recipes["ingredient_list"])

0

In [156]:
recipes[recipes["ingredient_list"].apply(len) == 0].shape

(0, 11)

In [157]:
recipes = recipes[
    recipes["ingredient_list"].apply(len) > 0
]

## Simpan Output

In [158]:
recipes.to_parquet("data/recipes_clean.parquet", index=False)

In [159]:
idf_df.to_parquet("data/ingredient_idf.parquet", index=False)

## Verifikasi Load

In [160]:
recipes = pd.read_parquet("data/recipes_clean.parquet")
recipes.head()

,Title,Title Cleaned,Category,Ingredients Cleaned,Steps,Loves,URL,Total Ingredients,Total Steps,ingredient_list,ingredient_count
0,Ayam Woku Manado,ayam woku manado,ayam,"ayam kampung potong , jeruk nipis , garam , ku...",1) Cuci bersih ayam dan tiriskan. Lalu peras j...,1,https://cookpad.com/id/resep/4473027-ayam-woku...,14,7,"[ayam, jeruk nipis, garam, kunyit, bawang mera...",14
1,Ayam goreng tulang lunak,ayam goreng tulang lunak,ayam,"ayam dipotong , serai , daun jeruk , bawang pu...","1) Haluskan bumbu2nya (BaPut, ketumbar, kemiri...",1,https://cookpad.com/id/resep/4471956-ayam-gore...,11,5,"[ayam, serai, daun jeruk, bawang putih, biji k...",11
2,Ayam cabai kawin,ayam cabai kawin,ayam,"ayam , cabai hijau , cabai merah rawit , bawan...",1) Panaskan minyak di dalam wajan. Setelah min...,2,https://cookpad.com/id/resep/4473057-ayam-caba...,10,3,"[ayam, cabai hijau, cabai merah rawit, bawang ...",10
3,Ayam Geprek,ayam geprek,ayam,"daging ayam fillet , gula garam , tepung ayam ...",1) Goreng ayam seperti ayam krispi\n2) Ulek se...,10,https://cookpad.com/id/resep/4473023-ayam-geprek,7,3,"[ayam, garam, ayam, daun kemangi, kol, timun, ...",9
4,Minyak Ayam,minyak ayam,ayam,"kulit ayam & lemaknya , bawang putih , cincang...",1) Cuci bersih kulit ayam. Sisihkan\n2) Ambil ...,4,https://cookpad.com/id/resep/4427438-minyak-ayam,5,6,"[ayam, bawang putih, jahe, minyak goreng, biji...",5


In [161]:
ingredient_idf = pd.read_parquet("data/ingredient_idf.parquet")
ingredient_idf.head()

,ingredient,idf
0,cabai,1.110865
1,kemiri,1.739296
2,jeruk nipis,2.982769
3,air,0.936910
4,daun salam,1.590548
